### READ BEFORE USING:
Only three cells should be modified by the user, 
- the stochastic process ([section 2](#set-model)), 
- the identifiability scenario ([section 3](#set-structural-identifiability-scenario)), and 
- the structural identifiability analysis ([section 5.3](#diagnose)).

## 1. Import packages

In [30]:
using Catalyst
using MomentClosure
using Latexify

using LinearAlgebra

In [31]:
# # Install packages (just necessary for Google Colab)
# import Pkg
# Pkg.add("Catalyst")
# Pkg.add("MomentClosure")
# Pkg.add("Latexify")

## 2. Set model

In [32]:
# Uncomment the desired model and comment out the others, or define a new model using the style of the e****xamples
# Avoid using μ as parameter, it is used by MomentClosure.jl to indicate moments.

# # Birth-death (live and dead cells)
# rn = @reaction_network begin
#     @parameters b d
#     @species nₗ(t) n(t)
#         b, nₗ → 2nₗ
#         d, nₗ → n
# end

# # 2 consumers, 2 resources model ("biotic" resources)
# rn = @reaction_network begin
#     @species r₁(t) r₂(t) n₁(t) n₂(t)
#     # without loss of generality assume the cell yield of consumption ϵ = 1 cell, thus cᵢⱼ units are (1 / resource * cell)
#         (ρ₁-i₁*r₁,ρ₂-i₂*r₂), (r₁,r₂) → (2r₁,2r₂)
#         (d₁,d₂), (n₁,n₂) → 0
#         (c₁₁,c₁₂), (n₁+r₁,n₁+r₂) → 2n₁
#         (c₂₁,c₂₂), (n₂+r₁,n₂+r₂) → 2n₂
# end

# # 2 consumers, 2 resources model ("abiotic" resources)
# rn = @reaction_network begin
#     @species r₁(t) r₂(t) n₁(t) n₂(t)
#      # without loss of generality assume the cell yield of consumption ϵ = 1 cell, thus cᵢⱼ units are (1 / resource * cell)
#         (ρ₁,ρ₂), 0 → (r₁,r₂)
#         (d₁,d₂), (n₁,n₂) → 0
#         (c₁₁,c₁₂), (n₁+r₁,n₁+r₂) → 2n₁
#         (c₂₁,c₂₂), (n₂+r₁,n₂+r₂) → 2n₂
# end

# # 6 consumers, 6 resources model ("abiotic" resources)
# rn = @reaction_network begin
#     @species r₀(t) r₁(t) r₂(t) r₃(t) r₄(t) r₅(t) n₀(t) n₁(t) n₂(t) n₃(t) n₄(t) n₅(t)
#         (ρ₀,ρ₁,ρ₂,ρ₃,ρ₄,ρ₅), 0 → (r₀,r₁,r₂,r₃,r₄,r₅)
#         (d₀,d₁,d₂,d₃,d₄,d₅), (n₀,n₁,n₂,n₃,n₄,n₅) → 0
#         (c₀₀,c₀₁,c₀₂,c₀₃,c₀₄,c₀₅), (n₀+r₀,n₀+r₁,n₀+r₂,n₀+r₃,n₀+r₄,n₀+r₅) → 2n₀
#         (c₁₀,c₁₁,c₁₂,c₁₃,c₁₄,c₁₅), (n₁+r₀,n₁+r₁,n₁+r₂,n₁+r₃,n₁+r₄,n₁+r₅) → 2n₁
#         (c₂₀,c₂₁,c₂₂,c₂₃,c₂₄,c₂₅), (n₂+r₀,n₂+r₁,n₂+r₂,n₂+r₃,n₂+r₄,n₂+r₅) → 2n₂
#         (c₃₀,c₃₁,c₃₂,c₃₃,c₃₄,c₃₅), (n₃+r₀,n₃+r₁,n₃+r₂,n₃+r₃,n₃+r₄,n₃+r₅) → 2n₃
#         (c₄₀,c₄₁,c₄₂,c₄₃,c₄₄,c₄₅), (n₄+r₀,n₄+r₁,n₄+r₂,n₄+r₃,n₄+r₄,n₄+r₅) → 2n₄
#         (c₅₀,c₅₁,c₅₂,c₅₃,c₅₄,c₅₅), (n₅+r₀,n₅+r₁,n₅+r₂,n₅+r₃,n₅+r₄,n₅+r₅) → 2n₅
# end

# Lotka-Volterra 2 species (carrying capacity in growth)
rn = @reaction_network begin
    @species n₀(t) n₁(t)
        (b₀-i₀₀*n₀,b₁-i₁₁*n₁), (n₀,n₁) → (2n₀,2n₁)
        (i₀₁), (n₀+n₁) → n₀
        (i₁₀), (n₁+n₀) → n₁
end

# # Lotka-Volterra 3 species (carrying capacity in growth)
# rn = @reaction_network begin
#     @species n₀(t) n₁(t) n₂(t)
#         (b₀-i₀₀*n₀,b₁-i₁₁*n₁,b₂-i₂₂*n₂), (n₀,n₁,n₂) → (2n₀,2n₁,2n₂)
#         (i₀₁,i₀₂), (n₀+n₁,n₀+n₂) → n₀
#         (i₁₀,i₁₂), (n₁+n₀,n₁+n₂) → n₁
#         (i₂₀,i₂₁), (n₂+n₀,n₂+n₁) → n₂
# end

# # Lotka-Volterra 6 species (carrying capacity in growth)
# rn = @reaction_network begin
#     @species n₀(t) n₁(t) n₂(t) n₃(t) n₄(t) n₅(t)
#         (b₀-i₀₀*n₀,b₁-i₁₁*n₁,b₂-i₂₂*n₂,b₃-i₃₃*n₃,b₄-i₄₄*n₄,b₅-i₅₅*n₅), (n₀,n₁,n₂,n₃,n₄,n₅) → (2n₀,2n₁,2n₂,2n₃,2n₄,2n₅)
#         (i₀₁,i₀₂,i₀₃,i₀₄,i₀₅), (n₀+n₁,n₀+n₂,n₀+n₃,n₀+n₄,n₀+n₅) → n₀
#         (i₁₀,i₁₂,i₁₃,i₁₄,i₁₅), (n₁+n₀,n₁+n₂,n₁+n₃,n₁+n₄,n₁+n₅) → n₁
#         (i₂₀,i₂₁,i₂₃,i₂₄,i₂₅), (n₂+n₀,n₂+n₁,n₂+n₃,n₂+n₄,n₂+n₅) → n₂
#         (i₃₀,i₃₁,i₃₂,i₃₄,i₃₅), (n₃+n₀,n₃+n₁,n₃+n₂,n₃+n₄,n₃+n₅) → n₃
#         (i₄₀,i₄₁,i₄₂,i₄₃,i₄₅), (n₄+n₀,n₄+n₁,n₄+n₂,n₄+n₃,n₄+n₅) → n₄
#         (i₅₀,i₅₁,i₅₂,i₅₃,i₅₄), (n₅+n₀,n₅+n₁,n₅+n₂,n₅+n₃,n₅+n₄) → n₅
# end

# # Michaelis-Menten
# rn = @reaction_network begin
#     @species e(t) s(t) es(t) p(t)
#         (k₊, k₋), e + s <--> es
#         (kₚ), es → e + p
# end

# # Susceptible-Infected-Recovered model (constant population size)
# rn = @reaction_network begin
#     @species s(t) i(t) r(t)
#         β, s + i → 2i
#         γ, i → r
# end

# # Microbial cross-feeding (CoSMO)
# rn = @reaction_network begin
#     @species A(t) L(t) a(t) l(t)
#     # @parameters bₐ bₗ dₐ dₗ cₗ cₐ rₐ rₗ
#         (bₐ, bₗ), (A+cₗl, L+cₐa) → (2A, 2L)
#         (dₐ, dₗ), (A, L) → (0, 0)
#         (rₐ, rₗ), (A, L) → (A+a, L+l)
# end

Model ##ReactionSystem#267:
Unknowns (2): see unknowns(##ReactionSystem#267)
  n₀(t)
  n₁(t)
Parameters (6): see parameters(##ReactionSystem#267)
  b₀
  i₀₀
  b₁
  i₁₁
  i₀₁
  i₁₀

## 3. Set Structural Identifiability scenario

In [33]:
# Print variables and their index in the model
speciesmap(rn)

Dict{SymbolicUtils.BasicSymbolic{Real}, Int64} with 2 entries:
  n₁(t) => 2
  n₀(t) => 1

In [34]:
# Establish settings for the Structuctural Identifiability analysis
hidden_variables = [2]     # List hidden variables. 
hidden_init_cond = [2]     # List hidden initial conditions

order_of_moments = 1            # Maximum order of moments.
moments_type = "raw"            # Options are: raw, central
closure = "none"                # Moment closure. Options are: none, zero, normal, poisson, log-normal, gamma, and others.

# Print warning if all variables are hidden.
if length(hidden_variables) >= length(species(rn))
    print("ERROR! Not all variables can be hidden.")
end

## 4. Get set of ODEs

In [35]:
# Derive moments ODEs
if moments_type == "raw"
    mom_eqs = generate_raw_moment_eqs(rn, order_of_moments, combinatoric_ratelaws=false)
elseif moments_type == "central"
    mom_eqs = generate_central_moment_eqs(rn, order_of_moments, combinatoric_ratelaws=false)
end
# latexify(mom_eqs) #|> print

# Use moment closure (zero, normal, poisson, log-normal, gamma, ...)
if closure == "none"
    closed_mom_eqs = mom_eqs
else
    closed_mom_eqs = moment_closure(mom_eqs, closure)
end
latexify(closed_mom_eqs) #|> print # this is to print `latex`

L"\begin{align*}
\frac{d\mu_{1 0}}{dt} &= b_0 \mu_{1 0} - i_{0 0} \mu_{2 0} - i_{1 0} \mu_{1 1} \\
\frac{d\mu_{0 1}}{dt} &= b_1 \mu_{0 1} - i_{0 1} \mu_{1 1} - i_{1 1} \mu_{0 2}
\end{align*}
"

In [36]:
# Extract parameters, moments and equations from the closed system of ODEs
params = parameters(closed_mom_eqs.odes)
moments = unknowns(closed_mom_eqs.odes)
moments_odes = equations(closed_mom_eqs.odes)

# Count number of parameters and ODEs
n_params = length(params)
n_moments = length(moments)
n_equations = length(moments_odes)

# Print summary
println("Number of parameters: \t", n_params)
println("Number of moments: \t", n_moments)
println("Number of ODEs: \t", n_equations)

Number of parameters: 	6
Number of moments: 	5
Number of ODEs: 	2


## 5. Benchmark: StructuralIdentifiability.jl

### 5.1 Import packages

In [37]:
# Import packages
using StructuralIdentifiability
using ModelingToolkit
using Logging

### 5.2 Set model

In [38]:
# Check if a hidden variable appears in a symbolic moment (=0 is absence, >0 presence)
subindx_dict = Dict('₀'=>0, '₁'=>1, '₂'=>2, '₃'=>3, '₄'=>4, '₅'=>5, '₆'=>6, '₇'=>7, '₈'=>8, '₉'=>9)
function hidden_involvement(moment, hidden_variables)
    subindex = [subindx_dict[x] for x in string(moment)[3 .* hidden_variables]]
    return sum(subindex) != 0
end

hidden_involvement (generic function with 1 method)

In [39]:
# Define symbolic variables
@independent_variables t
@variables y[1:n_moments]

# Trasform ODESystem from MomentClosure.jl to a ModelingToolkit.jl System
model_sijl = System(moments_odes, t, name = :model_sijl)

# Specify vector of observed moments and initial conditions
observed = [y[i] ~ moments[i] for i in 1:n_moments]
init_cond = isempty(hidden_init_cond) ? moments : [moments[i] for i in 1:n_moments if !hidden_involvement(moments[i], hidden_init_cond)]
if !isempty(hidden_variables)
    observed = [y[i] ~ moments[i] for i in 1:n_moments if !hidden_involvement(moments[i], hidden_variables)]
end

println(model_sijl)
println("observed: ", observed)
println("init. cond.: ", init_cond)

Model model_sijl:
Equations (2):
  2 standard: see equations(model_sijl)
Unknowns (5): see unknowns(model_sijl)
  μ₁₀(t)
  μ₀₁(t)
  μ₂₀(t)
  μ₁₁(t)
  μ₀₂(t)
Parameters (6): see parameters(model_sijl)
  b₀
  i₀₀
  i₁₀
  i₁₁
  b₁
  i₀₁
observed: Equation[y[1] ~ μ₁₀(t), y[3] ~ μ₂₀(t)]
init. cond.: SymbolicUtils.BasicSymbolic{Real}[μ₁₀(t), μ₂₀(t)]


### 5.3 Diagnose

In [40]:
# StructuralIdentifiability.jl assesment
# Uncomment the function that you want to see printed, comment the other functions

# Diagnose local identifiability only
# @time SIjl_output = assess_local_identifiability(model_sijl; measured_quantities = observed, prob_threshold=0.99, type=:SE, loglevel=Logging.Error) #funcs_to_check

# Diagnose global and local identifiability
@time SIjl_output = assess_identifiability(model_sijl, measured_quantities = observed, known_ic = init_cond, prob_threshold=0.99, loglevel=Logging.Error)

# Find identifiable functions (i.e. Groebner bases of the coefficients in the input-output equation)
# @time SIjl_output = find_identifiable_functions(model_sijl; with_states=true, measured_quantities = observed, known_ic = init_cond, prob_threshold=0.99, loglevel=Logging.Error)

  0.237071 seconds (2.51 M allocations: 120.881 MiB, 21.22% gc time)


OrderedCollections.OrderedDict{SymbolicUtils.BasicSymbolic{Real}, Symbol} with 8 entries:
  μ₁₀(0) => :globally
  μ₀₁(0) => :nonidentifiable
  b₀     => :globally
  i₀₀    => :globally
  i₁₀    => :globally
  i₁₁    => :nonidentifiable
  b₁     => :nonidentifiable
  i₀₁    => :nonidentifiable